# Normalizacao do Relatorio de Despesas do Sistema ATUA (03/2026)

Este notebook le o arquivo `Relatorio_Despesas_Sistema-ATUA_03.xls` (sistema ATUA, GSL Logistica) e converte para o layout do fechamento SAGI (`FECHAMENTO_ODBC_2026_03.xlsx`).

Diferente do ATUA, o SAGI classifica a GSL na divisao **TRANSMOVE GSL (1.4)** com quatro filiais e dois departamentos analiticos (TRANSPORTE / ADMINISTRATIVO). Este notebook:

1. Le os dados da aba `base`
2. Converte o Centro de Custo do ATUA (`cd_unidade` + `cd_centro_custo`) para o padrao SAGI
3. Converte o Plano de Contas do ATUA (`cd_historico` / `nm_historico`) para o padrao SAGI
4. Gera um Excel no layout FECHAMENTO_ODBC

In [14]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ARQUIVO_ENTRADA = ATUA_DIR / "Relatorio_Despesas_Sistema-ATUA_03.xls"
ARQUIVO_MODELO_FECHAMENTO = REFS_DIR / "FECHAMENTO_ODBC_2026_03.xlsx"
ARQUIVO_SAIDA = ATUA_DIR / "ATUA_despesas_fechamento_03-2026.xlsx"

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo ATUA nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

COLUNAS_INTERESSE = [
    "dt_lancamento",
    "nm_pessoa_favorecido",
    "cd_historico",
    "nm_historico",
    "nm_pessoa_filial",
    "dt_lancamento_",
    "vl_lancamento",
    "ds_complemento",
    "cd_centro_custo",
    "nm_centro_custo",
    "cd_unidade",
    "nm_unidade",
]

df_atua = pd.read_excel(ARQUIVO_ENTRADA, sheet_name="base", header=0, engine="xlrd", dtype=object)
df_atua = df_atua[COLUNAS_INTERESSE].copy()

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Linhas lidas: {len(df_atua)}")
print(f"Colunas usadas: {df_atua.columns.tolist()}")
df_atua.head(5)

Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\Relatorio_Despesas_Sistema-ATUA_03.xls
Linhas lidas: 96
Colunas usadas: ['dt_lancamento', 'nm_pessoa_favorecido', 'cd_historico', 'nm_historico', 'nm_pessoa_filial', 'dt_lancamento_', 'vl_lancamento', 'ds_complemento', 'cd_centro_custo', 'nm_centro_custo', 'cd_unidade', 'nm_unidade']


,dt_lancamento,nm_pessoa_favorecido,cd_historico,nm_historico,nm_pessoa_filial,dt_lancamento_,vl_lancamento,ds_complemento,cd_centro_custo,nm_centro_custo,cd_unidade,nm_unidade
0,2026-03-02 10:49:57.294434,TERRA ASSESSORIA IMOBILIARIA LTDA,16,ALUGUEL E CONDOMINIOS,GSL DOURADOS,2026-03-02 08:56:26,1003.5,ALUGUEL 03/2026 - GSL 03,100,ADMINISTRATIVO/COMERCIAL,16,DOURADOS ADMINISTRATIVO/COMERCIAL
1,2026-03-02 11:09:55.683029,"RSE COMERCIO DE FERRO, ACO E LOCACAO DE EQUIPA...",16,ALUGUEL E CONDOMINIOS,GSL PRUDENTE,2026-03-02 08:56:26,1671.35,NaN,100,ADMINISTRATIVO/COMERCIAL,9,PRUDENTE ADMINISTRATIVO/COMERCIAL
2,2026-03-02 16:10:48.006238,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,46,IMPOSTOS E TAXAS DIVERSAS,GSL PRUDENTE,2026-03-02 08:56:26,140.13,REFERENTE TRANSACOES FATURADAS,102,FROTA TERCEIRO PRUDENTE,8,FROTA TERCEIRO PRUDENTE
3,2026-03-02 16:14:16.656434,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,46,IMPOSTOS E TAXAS DIVERSAS,GSL PRUDENTE,2026-03-02 08:56:26,252.25,REFERENTE TRANSACOES FATURADAS,102,FROTA TERCEIRO PRUDENTE,8,FROTA TERCEIRO PRUDENTE
4,2026-03-02 16:17:11.081495,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,46,IMPOSTOS E TAXAS DIVERSAS,GSL PRUDENTE,2026-03-02 08:56:26,105.69,REFERENTE TRANSACOES FATURADAS,102,FROTA TERCEIRO PRUDENTE,8,FROTA TERCEIRO PRUDENTE


## Mapa de Centros de Custo (ATUA -> SAGI)

O ATUA classifica o CC com dois codigos: `cd_unidade` (filial) + `cd_centro_custo` (departamento).
No SAGI, a GSL e a divisao **1.4 TRANSMOVE GSL**, com 4 filiais:

| cd_unidade ATUA | Filial ATUA | Filial SAGI | Codigo SAGI nivel 3 |
|---|---|---|---|
| 8 | FROTA TERCEIRO PRUDENTE | PRESIDENTE PRUDENTE | 1.4.1 |
| 9 | PRUDENTE ADMINISTRATIVO/COMERCIAL | PRESIDENTE PRUDENTE | 1.4.1 |
| 12 | MARINGA ADMINISTRATIVO/COMERCIAL | MARINGA | 1.4.3 |
| 16 | DOURADOS ADMINISTRATIVO/COMERCIAL | DOURADOS | 1.4.2 |
| 26 | BARUERI ADMINISTRATIVO/COMERCIAL | BARUERI | 1.4.4 |

E dois departamentos analiticos:

| cd_centro_custo ATUA | Nome ATUA | Departamento SAGI | Sufixo codigo |
|---|---|---|---|
| 81 | Transporte | TRANSPORTE | .1 |
| 95 | Sucata | TRANSPORTE | .1 |
| 100 | ADMINISTRATIVO/COMERCIAL | ADMINISTRATIVO | .2 |
| 102 | FROTA TERCEIRO PRUDENTE | TRANSPORTE | .1 |

In [15]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

def _int_str(v) -> str:
    """Converte floats como 9.0 para '9' (o pandas as vezes le inteiros como float)."""
    if pd.isna(v):
        return ""
    try:
        f = float(v)
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return _str(v)

print("=== Valores unicos de filial/unidade no ATUA ===")
print(df_atua[["cd_unidade", "nm_unidade"]].drop_duplicates().sort_values("cd_unidade").to_string(index=False))
print()
print("=== Valores unicos de departamento no ATUA ===")
print(df_atua[["cd_centro_custo", "nm_centro_custo"]].drop_duplicates().sort_values("cd_centro_custo").to_string(index=False))

MAPA_FILIAL = {
    "8":  {"n3_cod": "1.4.1", "n3_desc": "PRESIDENTE PRUDENTE", "filial_saida": "GSL PRUDENTE"},
    "9":  {"n3_cod": "1.4.1", "n3_desc": "PRESIDENTE PRUDENTE", "filial_saida": "GSL PRUDENTE"},
    "12": {"n3_cod": "1.4.3", "n3_desc": "MARINGA",             "filial_saida": "GSL MARINGA"},
    "16": {"n3_cod": "1.4.2", "n3_desc": "DOURADOS",            "filial_saida": "GSL DOURADOS"},
    "26": {"n3_cod": "1.4.4", "n3_desc": "BARUERI",             "filial_saida": "GSL BARUERI"},
}

MAPA_DEPARTAMENTO = {
    "81":  {"sufixo": "1", "desc": "TRANSPORTE"},
    "95":  {"sufixo": "1", "desc": "TRANSPORTE"},
    "100": {"sufixo": "2", "desc": "ADMINISTRATIVO"},
    "102": {"sufixo": "1", "desc": "TRANSPORTE"},
}

def mapear_cc(cd_unidade, cd_centro_custo) -> dict | None:
    """Retorna o codigo SAGI completo + toda a hierarquia ou None se nao for mapeavel."""
    uni = _int_str(cd_unidade)
    dep = _int_str(cd_centro_custo)
    filial = MAPA_FILIAL.get(uni)
    depto = MAPA_DEPARTAMENTO.get(dep)
    if not filial or not depto:
        return None
    cod_n4 = f"{filial['n3_cod']}.{depto['sufixo']}"
    return {
        "n1_cod": "1",
        "n1_desc": "DESPESA",
        "n2_cod": "1.4",
        "n2_desc": "TRANSMOVE GSL",
        "n3_cod": filial["n3_cod"],
        "n3_desc": filial["n3_desc"],
        "n4_cod": cod_n4,
        "n4_desc": depto["desc"],
        "filial_saida": filial["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

ccs_nao_mapeados_cc = []
for _, r in df_atua[["cd_unidade", "cd_centro_custo", "nm_unidade", "nm_centro_custo"]].drop_duplicates().iterrows():
    if mapear_cc(r["cd_unidade"], r["cd_centro_custo"]) is None:
        ccs_nao_mapeados_cc.append(
            (_int_str(r["cd_unidade"]), _str(r["nm_unidade"]), _int_str(r["cd_centro_custo"]), _str(r["nm_centro_custo"]))
        )

if ccs_nao_mapeados_cc:
    print("\n[ALERTA] Centros de Custo nao mapeados:")
    for item in ccs_nao_mapeados_cc:
        print(f"  cd_unidade={item[0]} ({item[1]}) | cd_centro_custo={item[2]} ({item[3]})")
else:
    print("\nTodos os centros de custo foram mapeados com sucesso.")

=== Valores unicos de filial/unidade no ATUA ===
cd_unidade                        nm_unidade
         8           FROTA TERCEIRO PRUDENTE
         9 PRUDENTE ADMINISTRATIVO/COMERCIAL
        12  MARINGA ADMINISTRATIVO/COMERCIAL
        16 DOURADOS ADMINISTRATIVO/COMERCIAL
        26  BARUERI ADMINISTRATIVO/COMERCIAL

=== Valores unicos de departamento no ATUA ===
cd_centro_custo          nm_centro_custo
             81               Transporte
             95                   Sucata
            100 ADMINISTRATIVO/COMERCIAL
            102  FROTA TERCEIRO PRUDENTE

Todos os centros de custo foram mapeados com sucesso.


## Mapa de Plano de Contas (ATUA -> SAGI)

O ATUA usa `cd_historico` (codigo) e `nm_historico` (descricao) como plano de contas. Esta celula mapeia cada historico ATUA para o codigo oficial do SAGI (`02-Referencias/Plano de Contas.pdf`):

| cd_historico ATUA | nm_historico ATUA | Codigo SAGI | Descricao SAGI |
|---|---|---|---|
| 15 | ENERGIA ELETRICA | 7.5.2 | ENERGIA ELETRICA |
| 16 | ALUGUEL E CONDOMINIOS | 7.5.31 | ALUGUEL ADMINISTRATIVO |
| 17 | SEGURO DE CARGAS | 6.6.4 | SEGURO DE CARGAS |
| 23 | TARIFAS BANCARIAS | 7.5.22 | DESPESAS BANCARIAS |
| 26 | ASSESSORIAS E TELECONSULTAS | 7.5.9 | CONSULTORIA |
| 46 | IMPOSTOS E TAXAS DIVERSAS | 7.5.17 | TAXAS |
| 62 | FRETES PAGOS | 6.6.1 | FRETE DE TERCEIROS |
| 69 | HONORARIOS CONTABEIS | 7.5.7 | HONORARIOS CONTABEIS |
| 95 | ICMS | 7.4.12 | ICMS |
| 209 | PESSOAL - INSS PATRONAL | 7.3.3 | INSS |
| 229 | PESSOAL - PRO LABORE | 7.3.12 | PRO LABORE |
| 231 | DIESEL - PAMCARD | 7.1.4 | COMBUSTIVEL - DIESEL (POSTO) |

In [16]:
print("=== Valores unicos de Plano de Contas no ATUA ===")
print(df_atua[["cd_historico", "nm_historico"]].drop_duplicates().sort_values("cd_historico").to_string(index=False))

MAPA_PLANO_CONTAS = {
    "15":  {"cod": "7.5.2",  "desc": "ENERGIA ELETRICA"},
    "16":  {"cod": "7.5.31", "desc": "ALUGUEL ADMINISTRATIVO"},
    "17":  {"cod": "6.6.4",  "desc": "SEGURO DE CARGAS"},
    "23":  {"cod": "7.5.22", "desc": "DESPESAS BANCARIAS"},
    "26":  {"cod": "7.5.9",  "desc": "CONSULTORIA"},
    "46":  {"cod": "7.5.17", "desc": "TAXAS"},
    "62":  {"cod": "6.6.1",  "desc": "FRETE DE TERCEIROS"},
    "69":  {"cod": "7.5.7",  "desc": "HONORARIOS CONTABEIS"},
    "95":  {"cod": "7.4.12", "desc": "ICMS"},
    "209": {"cod": "7.3.3",  "desc": "INSS"},
    "229": {"cod": "7.3.12", "desc": "PRO LABORE"},
    "231": {"cod": "7.1.4",  "desc": "COMBUSTIVEL - DIESEL (POSTO)"},
}

def mapear_plano_contas(cd_historico) -> dict | None:
    key = _int_str(cd_historico)
    return MAPA_PLANO_CONTAS.get(key)

historicos_nao_mapeados = []
for _, r in df_atua[["cd_historico", "nm_historico"]].drop_duplicates().iterrows():
    if mapear_plano_contas(r["cd_historico"]) is None:
        historicos_nao_mapeados.append((_int_str(r["cd_historico"]), _str(r["nm_historico"])))

if historicos_nao_mapeados:
    print("\n[ALERTA] Planos de Contas nao mapeados:")
    for cod, desc in historicos_nao_mapeados:
        print(f"  cd_historico={cod} ({desc})")
else:
    print("\nTodos os planos de contas foram mapeados com sucesso.")

=== Valores unicos de Plano de Contas no ATUA ===
cd_historico                nm_historico
          15            ENERGIA ELETRICA
          16       ALUGUEL E CONDOMINIOS
          17            SEGURO DE CARGAS
          23           TARIFAS BANCARIAS
          26 ASSESSORIAS E TELECONSULTAS
          46   IMPOSTOS E TAXAS DIVERSAS
          62                FRETES PAGOS
          69        HONORARIOS CONTABEIS
          95                        ICMS
         209     PESSOAL - INSS PATRONAL
         229        PESSOAL - PRO LABORE
         231            DIESEL - PAMCARD

Todos os planos de contas foram mapeados com sucesso.


## Conversao para layout FECHAMENTO_ODBC

Regras de mapeamento aplicadas:

- `nm_pessoa_filial` -> `filial`
- `vl_lancamento` -> `valor_nf`, `valor_pago`, `valor_conta` (o ATUA nao distingue entre esses valores; usamos o mesmo para os tres campos)
- `dt_lancamento` -> `data_nf`
- `dt_lancamento_` -> `data_pagamento`
- `nm_pessoa_favorecido` -> `credor_forn_cli_func`
- `ds_complemento` -> `observacao`
- `cd_unidade` + `cd_centro_custo` -> `n1_cod_centro_custo` ate `n4_cod_centro_custo` + descricoes hierarquicas
- `cd_historico` -> `cod_conta` + `conta` + `cod_conta-descr`
- `Origem` = "ATUA"
- `Sistema` = "ATUA"

In [17]:
def _to_float(v):
    if pd.isna(v):
        return pd.NA
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace("R$", "").replace(" ", "")
    if not s:
        return pd.NA
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return pd.NA

def _fmt_brl(v):
    if pd.isna(v):
        return ""
    s = f"{float(v):,.2f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def _fmt_data(v) -> str:
    """Formata datetimes do pandas como dd/mm/aaaa. Se ja for string, mantem."""
    if pd.isna(v) or v == "":
        return ""
    try:
        ts = pd.to_datetime(v, errors="coerce")
        if pd.isna(ts):
            return str(v).strip()
        return ts.strftime("%d/%m/%Y")
    except Exception:
        return str(v).strip()

modelo_cols = pd.read_excel(ARQUIVO_MODELO_FECHAMENTO, nrows=0).columns.tolist()
print("Colunas do modelo FECHAMENTO:", modelo_cols)

linhas_saida = []
alertas_linhas_cc = []
alertas_linhas_pc = []

for i, row in df_atua.iterrows():
    cc_map = mapear_cc(row["cd_unidade"], row["cd_centro_custo"])
    pc_map = mapear_plano_contas(row["cd_historico"])

    if cc_map is None:
        alertas_linhas_cc.append({
            "idx": i,
            "cd_unidade": _int_str(row["cd_unidade"]),
            "nm_unidade": _str(row["nm_unidade"]),
            "cd_centro_custo": _int_str(row["cd_centro_custo"]),
            "nm_centro_custo": _str(row["nm_centro_custo"]),
        })
    if pc_map is None:
        alertas_linhas_pc.append({
            "idx": i,
            "cd_historico": _int_str(row["cd_historico"]),
            "nm_historico": _str(row["nm_historico"]),
        })

    valor = _to_float(row["vl_lancamento"])

    nova = {c: pd.NA for c in modelo_cols}
    nova["id"] = i + 1

    if cc_map is not None:
        nova["Segmento"] = cc_map["segmento"]
        nova["n1_cod_centro_custo"] = cc_map["n1_cod"]
        nova["n1_centro_custo"] = cc_map["n1_desc"]
        nova["n1_CC"] = f"{cc_map['n1_cod']} {cc_map['n1_desc']}"
        nova["n2_cod_centro_custo"] = cc_map["n2_cod"]
        nova["n2_centro_custo"] = cc_map["n2_desc"]
        nova["n2_CC"] = f"{cc_map['n2_cod']} {cc_map['n2_desc']}"
        nova["n3_cod_centro_custo"] = cc_map["n3_cod"]
        nova["n3_centro_custo"] = cc_map["n3_desc"]
        nova["n3_CC"] = f"{cc_map['n3_cod']} {cc_map['n3_desc']}"
        nova["n4_cod_centro_custo"] = cc_map["n4_cod"]
        nova["n4_centro_custo"] = cc_map["n4_desc"]
        nova["n4_CC"] = f"{cc_map['n4_cod']} {cc_map['n4_desc']}"
        nova["filial"] = cc_map["filial_saida"]
    else:
        nova["filial"] = _str(row["nm_pessoa_filial"])

    if pc_map is not None:
        nova["cod_conta"] = pc_map["cod"]
        nova["conta"] = pc_map["desc"]
        nova["cod_conta-descr"] = f"{pc_map['cod']} {pc_map['desc']}"

    nova["titulo"] = f"ATUA-{i+1}"
    nova["valor_nf"] = _fmt_brl(valor)
    nova["valor_pago"] = _fmt_brl(valor)
    nova["valor_conta"] = _fmt_brl(valor)
    nova["observacao"] = _str(row["ds_complemento"])
    nova["data_nf"] = _fmt_data(row["dt_lancamento"])
    nova["data_pagamento"] = _fmt_data(row["dt_lancamento_"])
    nova["credor_forn_cli_func"] = _str(row["nm_pessoa_favorecido"])
    nova["Origem"] = "ATUA"
    nova["Sistema"] = "ATUA"

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

print(f"\nLinhas geradas: {len(fechamento_df)}")
print(f"Linhas com CC nao mapeado: {len(alertas_linhas_cc)}")
print(f"Linhas com PC nao mapeado: {len(alertas_linhas_pc)}")
fechamento_df.head(10)

Colunas do modelo FECHAMENTO: ['id', 'Segmento', 'n1_cod_centro_custo', 'n1_centro_custo', 'n1_CC', 'n2_cod_centro_custo', 'n2_centro_custo', 'n2_CC', 'n3_cod_centro_custo', 'n3_centro_custo', 'n3_CC', 'n4_cod_centro_custo', 'n4_centro_custo', 'n4_CC', 'cod_conta', 'conta', 'cod_conta-descr', 'filial', 'titulo', 'valor_nf', 'valor_pago', 'valor_conta', 'observacao', 'data_nf', 'data_pagamento', 'cod_credor_forn_cli_func', 'credor_forn_cli_func', 'Origem', 'Sistema', 'Dados auxiliares', ' Valor Oficial ', 'DE-PARA1', 'DE-PARA2', 'CUSTEIO VARIÁVEL']

Linhas geradas: 96
Linhas com CC nao mapeado: 0
Linhas com PC nao mapeado: 0


,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,n3_CC,n4_cod_centro_custo,n4_centro_custo,n4_CC,cod_conta,...,valor_nf,valor_pago,valor_conta,observacao,data_nf,data_pagamento,cod_credor_forn_cli_func,credor_forn_cli_func,Origem,Sistema,Dados auxiliares,Valor Oficial,DE-PARA1,DE-PARA2,CUSTEIO VARIÁVEL
0,1,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.2,DOURADOS,1.4.2 DOURADOS,1.4.2.2,ADMINISTRATIVO,1.4.2.2 ADMINISTRATIVO,7.5.31,...,"1.003,50","1.003,50","1.003,50",ALUGUEL 03/2026 - GSL 03,02/03/2026,02/03/2026,<NA>,TERRA ASSESSORIA IMOBILIARIA LTDA,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.2,ADMINISTRATIVO,1.4.1.2 ADMINISTRATIVO,7.5.31,...,"1.671,35","1.671,35","1.671,35",,02/03/2026,02/03/2026,<NA>,"RSE COMERCIO DE FERRO, ACO E LOCACAO DE EQUIPA...",ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
2,3,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"140,13","140,13","140,13",REFERENTE TRANSACOES FATURADAS,02/03/2026,02/03/2026,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
3,4,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"252,25","252,25","252,25",REFERENTE TRANSACOES FATURADAS,02/03/2026,02/03/2026,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
4,5,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"105,69","105,69","105,69",REFERENTE TRANSACOES FATURADAS,02/03/2026,02/03/2026,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
5,6,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.5.17,...,"126,48","126,48","126,48",REFERENTE TRANSACOES FATURADAS,02/03/2026,02/03/2026,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
6,7,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.2,ADMINISTRATIVO,1.4.1.2 ADMINISTRATIVO,6.6.1,...,"199,00","199,00","199,00",,02/03/2026,02/03/2026,<NA>,NSTECH IP INSTITUICAO DE PAGAMENTO S.A.,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
7,8,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.1.4,...,"1.116,60","1.116,60","1.116,60","TRIB APROX R$: 96,03 FEDERAL, 204,34 ESTADUAL|...",04/03/2026,04/03/2026,<NA>,AUTO POSTO PHOENIX DE ITAPETININGA LTDA - ME,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
8,9,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.1.4,...,"1.525,00","1.525,00","1.525,00","RESUMO PAGAMENTO: 1525,03| - CARTA FRETE: R$ ...",04/03/2026,04/03/2026,<NA>,G10 - AUTO POSTO S.A,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>
9,10,TRANSMOVE GSL,1,DESPESA,1 DESPESA,1.4,TRANSMOVE GSL,1.4 TRANSMOVE GSL,1.4.1,PRESIDENTE PRUDENTE,1.4.1 PRESIDENTE PRUDENTE,1.4.1.1,TRANSPORTE,1.4.1.1 TRANSPORTE,7.1.4,...,"986,32","986,32","986,32",|ICMS MONOFASICO SOBRE COMBUSTIVEIS COBRADO AN...,04/03/2026,04/03/2026,<NA>,POSTO TREVO SERTANOPOLIS LTDA,ATUA,ATUA,<NA>,<NA>,<NA>,<NA>,<NA>


## Salvamento e relatorio

Gera o Excel final em `02-Referencias/ATUA/ATUA_despesas_fechamento_03-2026.xlsx` e imprime o relatorio de itens nao mapeados para revisao manual.

In [18]:
from openpyxl.styles import Font

arquivo_saida_exec = ARQUIVO_SAIDA
try:
    with pd.ExcelWriter(arquivo_saida_exec, engine="openpyxl") as writer:
        sheet_name = "ATUA_despesas_fechamento"
        fechamento_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font
except PermissionError:
    arquivo_saida_exec = ARQUIVO_SAIDA.with_name(ARQUIVO_SAIDA.stem + "_novo" + ARQUIVO_SAIDA.suffix)
    with pd.ExcelWriter(arquivo_saida_exec, engine="openpyxl") as writer:
        sheet_name = "ATUA_despesas_fechamento"
        fechamento_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font

print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
print(f"Linhas gravadas: {len(fechamento_df)}")

if ccs_nao_mapeados_cc:
    print("\n[REVISAO MANUAL] Centros de Custo nao mapeados:")
    for item in ccs_nao_mapeados_cc:
        print(f"  cd_unidade={item[0]} ({item[1]}) | cd_centro_custo={item[2]} ({item[3]})")

if historicos_nao_mapeados:
    print("\n[REVISAO MANUAL] Historicos (Plano de Contas) nao mapeados:")
    for cod, desc in historicos_nao_mapeados:
        print(f"  cd_historico={cod} ({desc})")

if not ccs_nao_mapeados_cc and not historicos_nao_mapeados:
    print("\nNenhum item pendente. Todos os centros de custo e planos de conta foram mapeados.")

Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\ATUA_despesas_fechamento_03-2026.xlsx
Linhas gravadas: 96

Nenhum item pendente. Todos os centros de custo e planos de conta foram mapeados.
